# 08 — Unified GeoPackage validation prototype

This notebook audits the current unified GeoPackage against the four processed child files. It then creates a separate corrected copy for testing. It never overwrites an input. The corrected copy is a prototype: move the validated rules into Notebook 07's final writer, then rebuild and rerun this audit before moving the methodology into `src`.

The NATCARB oil-and-gas unit link and BC unit-level assessment feature link are intentionally unknown. They must export as SQL `NULL`, not the string `<NA>`. BC pool formation comes from child `source_formation_name` only when every feature of the logical pool agrees. AER agreements and tracts remain tenure layers.


In [5]:
from pathlib import Path
import importlib.util
import json
import pyogrio
import geopandas as gpd

repo = Path.cwd()
if not (repo / 'notebooks').is_dir() and (repo.parent / 'notebooks').is_dir():
    repo = repo.parent
helper = repo / 'notebooks' / 'prototype_unification.py'
if not helper.exists():
    helper = Path.cwd() / 'prototype_unification.py'
spec = importlib.util.spec_from_file_location('prototype_unification', helper)
validation = importlib.util.module_from_spec(spec)
spec.loader.exec_module(validation)
print('Helper:', helper)
print('Source:', validation.UNIFIED)


Helper: c:\Users\aviga\Research\repos\canco2-storage\notebooks\prototype_unification.py
Source: C:\Users\aviga\Research\repos\canco2-storage\data\processed\compiled dbs\canada_geological_storage_unified.gpkg


In [2]:
before = validation.audit(validation.UNIFIED)
print(json.dumps({'blocking_findings': validation.blocking_findings(before), 'checks': before}, indent=2))


{
  "blocking_findings": [
    "literal_na_feature_parents=1358",
    "literal_na_assessment_unit_parents=1358",
    "literal_na_assessment_feature_parents=1263",
    "feature_unit_orphans=1358",
    "assessment_unit_orphans=1358",
    "assessment_feature_orphans=1263",
    "bc_pool_units_missing_formation=1238",
    "tenure_agreements does not match its AER child count",
    "tenure_tracts does not match its AER child count"
  ],
  "checks": {
    "integrity": "ok",
    "storage_units_rows": 1374,
    "storage_units_duplicate_ids": 0,
    "storage_units_missing_ids": 0,
    "storage_features_rows": 34839,
    "storage_features_duplicate_ids": 0,
    "storage_features_missing_ids": 0,
    "storage_assessments_rows": 34610,
    "storage_assessments_duplicate_ids": 0,
    "storage_assessments_missing_ids": 0,
    "natcarb_feature_attributes_rows": 28790,
    "natcarb_feature_attributes_duplicate_ids": 0,
    "natcarb_feature_attributes_missing_ids": 0,
    "literal_na_feature_parents": 1

In [3]:
output_dir = repo / 'data' / 'processed' / 'validation_prototypes'
output = output_dir / 'canada_geological_storage_validated_prototype.gpkg'
if not output.exists():
    validation.repair(output)
else:
    print('Existing prototype found; audit it without overwriting:', output)
after = validation.audit(output)
failures = validation.blocking_findings(after)
print(json.dumps({'path': str(output), 'blocking_findings': failures, 'checks': after}, indent=2))
assert not failures, failures
print('GIS layers:', pyogrio.list_layers(output))


{
  "path": "c:\\Users\\aviga\\Research\\repos\\canco2-storage\\data\\processed\\validation_prototypes\\canada_geological_storage_validated_prototype.gpkg",
  "blocking_findings": [],
  "checks": {
    "integrity": "ok",
    "storage_units_rows": 1374,
    "storage_units_duplicate_ids": 0,
    "storage_units_missing_ids": 0,
    "storage_features_rows": 34839,
    "storage_features_duplicate_ids": 0,
    "storage_features_missing_ids": 0,
    "storage_assessments_rows": 34610,
    "storage_assessments_duplicate_ids": 0,
    "storage_assessments_missing_ids": 0,
    "natcarb_feature_attributes_rows": 28790,
    "natcarb_feature_attributes_duplicate_ids": 0,
    "natcarb_feature_attributes_missing_ids": 0,
    "literal_na_feature_parents": 0,
    "literal_na_assessment_unit_parents": 0,
    "literal_na_assessment_feature_parents": 0,
    "feature_unit_orphans": 0,
    "assessment_unit_orphans": 0,
    "assessment_feature_orphans": 0,
    "natcarb_attribute_orphans": 0,
    "bc_missing_un

## Boundary of this validation

The strict checks establish ID uniqueness, child-feature coverage, BC and Atlantic parent links, BC formation transfer, AER tenure coverage, and SQL link validity. They do not establish NATCARB reservoir-to-zone relationships: the selected Canadian oil-and-gas child rows lack reservoir names and numbers. They also do not validate the earlier shapefile compilation or every geometry clipping decision. An authoritative field/reservoir/formation crosswalk and a separate spatial membership audit are required for those claims.

Before using the result as silver data, implement the writer in Notebook 07, rebuild from the child GeoPackages, and rerun this notebook on the fresh output. Give bronze records immutable source identity `(dataset, layer/file, source version, source UID)` and store a persistent source-to-root crosswalk; never use local GeoPackage `fid` as a root identifier.


In [6]:
gdf = gpd.read_file(output, layer="storage_features")

print(gdf.geom_type.value_counts(dropna=False))
print(gdf.crs)
print(gdf.geometry.isna().sum())
print(gdf.geometry.is_valid.value_counts(dropna=False))

MultiPolygon          34814
Polygon                  24
GeometryCollection        1
Name: count, dtype: int64
EPSG:3978
0
True    34839
Name: count, dtype: int64


In [7]:
geometry_collection = gdf[gdf.geom_type == "GeometryCollection"].copy()

print(
    geometry_collection[
        [
            "storage_feature_id",
            "source_dataset",
            "storage_unit_id",
        ]
    ]
)

geom = geometry_collection.geometry.iloc[0]

print(geom)
print([part.geom_type for part in geom.geoms])

                    storage_feature_id source_dataset  \
1313  gbc_aquifer_flatrock_monias_f001   gbc_ne_atlas   

                  storage_unit_id  
1313  gbc_aquifer_flatrock_monias  
GEOMETRYCOLLECTION (MULTIPOLYGON (((-1493746.8719325992 1071446.1641436063, -1493709.9421452987 1071404.973167184, -1493725.1029445915 1071192.57582561, -1493746.8719325992 1071446.1641436063)), ((-1540861.1179437216 1178287.588355675, -1540847.4750169322 1178286.683582792, -1540833.751398924 1178286.0479642567, -1540819.9454836773 1178285.6814219707, -1540806.055321477 1178285.583900476, -1540792.079019886 1178285.7553861865, -1540778.0145585022 1178286.195903608, -1540763.8612524513 1178286.9054344373, -1540749.6154840938 1178287.8841205458, -1540735.2749432456 1178289.1321202763, -1540720.837148499 1178290.649649248, -1540706.2997394837 1178292.4369495602, -1540691.659739579 1178294.4943812487, -1540676.9142749845 1178296.8223477278, -1540662.0602367718 1178299.421339404, -1540647.1172768823 1178302

In [9]:
gdf["geometry_type"] = gdf.geom_type

display(
    gdf.groupby(["source_dataset", "geometry_type"])
       .size()
       .rename("count")
       .reset_index()
)

,source_dataset,geometry_type,count
0,gbc_ne_atlas,GeometryCollection,1
1,gbc_ne_atlas,MultiPolygon,1313
2,gbc_ne_atlas,Polygon,24
3,gsc_atlantic,MultiPolygon,4711
4,natcarb_doe,MultiPolygon,28790


In [10]:
bc_gdf = gpd.read_file(
    validation.BC,
    layer="aquifer_features"
)

row = bc_gdf[
    bc_gdf["storage_feature_id"] == "gbc_aquifer_flatrock_monias_f001"
]

print(row.geom_type)
print(row.geometry.iloc[0])
print([part.geom_type for part in row.geometry.iloc[0].geoms]
      if row.geometry.iloc[0].geom_type == "GeometryCollection"
      else None)

4    GeometryCollection
dtype: str
GEOMETRYCOLLECTION (MULTIPOLYGON (((-1493746.8719325992 1071446.1641436063, -1493709.9421452987 1071404.973167184, -1493725.1029445915 1071192.57582561, -1493746.8719325992 1071446.1641436063)), ((-1540861.1179437216 1178287.588355675, -1540847.4750169322 1178286.683582792, -1540833.751398924 1178286.0479642567, -1540819.9454836773 1178285.6814219707, -1540806.055321477 1178285.583900476, -1540792.079019886 1178285.7553861865, -1540778.0145585022 1178286.195903608, -1540763.8612524513 1178286.9054344373, -1540749.6154840938 1178287.8841205458, -1540735.2749432456 1178289.1321202763, -1540720.837148499 1178290.649649248, -1540706.2997394837 1178292.4369495602, -1540691.659739579 1178294.4943812487, -1540676.9142749845 1178296.8223477278, -1540662.0602367718 1178299.421339404, -1540647.1172768823 1178302.287341697, -1540632.0593573255 1178305.424777124, -1540616.882838544 1178308.8344034287, -1540601.5837485462 1178312.517112439, -1540586.1583650718 117